# Part D – Local LLM and Data Privacy

Run an LLM **locally** without using any cloud APIs.

**Tool**: Hugging Face local inference with quantization (4-bit) to fit on consumer GPU.

In [1]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
print(f"Device: {device}")

d:\college\VI-sem\DL\lab-task-3\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA GeForce RTX 5070 Laptop GPU (8.0 GB)
Device: cuda


## Step 1: Load a Local LLM with 4-bit Quantization

We use **TinyLlama-1.1B-Chat** — a small but capable chat model that runs entirely locally.
4-bit quantization reduces memory from ~2.2GB (fp16) to ~0.6GB.

In [2]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

num_params = sum(p.numel() for p in model.parameters())
print(f"Model: {MODEL_NAME}")
print(f"Parameters: {num_params / 1e9:.2f}B")
print("Quantization: 4-bit NF4")
print("Model loaded entirely locally -- no cloud API used.")

`torch_dtype` is deprecated! Use `dtype` instead!


Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Parameters: 0.62B
Quantization: 4-bit NF4
Model loaded entirely locally -- no cloud API used.


## Step 2: Local Inference — No Internet Required

In [3]:
def chat_local(prompt, max_new_tokens=200):
    # Use tokenizer's built-in chat template
    messages = [
        {"role": "system", "content": "You are a helpful agriculture assistant."},
        {"role": "user", "content": prompt},
    ]
    chat_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - start_time
    num_tokens = outputs.shape[1] - inputs["input_ids"].shape[1]

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    return response.strip(), elapsed, num_tokens


# Test queries
test_queries = [
    "What is the best fertilizer for rice cultivation?",
    "How do I prevent fungal infections in tomato plants?",
    "What are the benefits of drip irrigation?",
    "Explain crop rotation and its importance.",
    "What is the ideal soil pH for growing potatoes?",
]

print("=" * 60)
print("LOCAL LLM INFERENCE (No Cloud API)")
print("=" * 60)
for query in test_queries:
    response, elapsed, num_tokens = chat_local(query)
    tokens_per_sec = num_tokens / elapsed if elapsed > 0 else 0
    print(f"\nQ: {query}")
    print(f"A: {response}")
    print(f"   [{num_tokens} tokens in {elapsed:.2f}s = {tokens_per_sec:.1f} tok/s]")
    print("-" * 50)

LOCAL LLM INFERENCE (No Cloud API)

Q: What is the best fertilizer for rice cultivation?
A: The best fertilizer for rice cultivation depends on the type of rice you want to grow. Here are some commonly used fertilizers for rice:

1. Nitrogen-rich fertilizers: These fertilizers contain nitrogen, which helps to promote growth of the rice plants. Some examples of nitrogen-rich fertilizers are ammonium nitrate, nitric acid, and ammonium sulfate.

2. Potassium-rich fertilizers: Potassium helps to regulate water uptake in the rice plants, which is essential for rice growth. Some examples of potassium-rich fertilizers are potassium nitrate, potassium sulfate, and potassium chloride.

3. Phosphorus-rich fertilizers: Phosphorus helps to promote growth of the rice plants by improving the nutrient uptake. Some examples
   [200 tokens in 9.68s = 20.7 tok/s]
--------------------------------------------------

Q: How do I prevent fungal infections in tomato plants?
A: Here are some tips to prevent f

## Step 3: Memory Usage Analysis

In [4]:
if torch.cuda.is_available():
    print("GPU Memory Usage:")
    print(f"  Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"  Reserved:  {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
    print(f"  Total GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    import psutil
    process_mem = psutil.Process().memory_info().rss / 1024**3
    print(f"CPU Memory used by process: {process_mem:.2f} GB")

GPU Memory Usage:
  Allocated: 0.73 GB
  Reserved:  1.00 GB
  Total GPU: 7.96 GB


## Step 4: Data Privacy Discussion

### Advantages of Local LLMs
1. **Complete data privacy** — no data leaves the local machine
2. **No internet dependency** — works offline after initial model download
3. **No API costs** — zero recurring costs after hardware investment
4. **Low latency** — no network round-trip delay
5. **Full control** — can customize, fine-tune, and modify the model freely

### Data Privacy Benefits
1. **Sensitive data stays local** — medical, financial, legal data never sent to third parties
2. **Regulatory compliance** — easier to comply with GDPR, HIPAA, etc.
3. **No data logging** — cloud providers may log prompts/responses
4. **Air-gapped deployment** — can run on completely isolated networks

### API-Based vs Local Deployment Comparison

| Aspect | API-Based (Cloud) | Local Deployment |
|---|---|---|
| **Data Privacy** | Data sent to provider | Data stays on-device |
| **Cost** | Pay-per-token (recurring) | One-time hardware cost |
| **Model Quality** | Latest large models (GPT-4, etc.) | Limited by local hardware |
| **Latency** | Network dependent (100-500ms+) | Very low (~50ms) |
| **Scalability** | Easily scalable | Limited by hardware |
| **Offline Use** | Not possible | Fully supported |
| **Customization** | Limited to API options | Full fine-tuning possible |
| **Compliance** | Depends on provider | Full control |
| **Setup Complexity** | Simple API key | Requires GPU + setup |
| **Model Size** | Unlimited | Constrained by VRAM/RAM |

In [5]:
print("=" * 60)
print("PART D SUMMARY")
print("=" * 60)
print(f"Model used: {MODEL_NAME}")
print(f"Quantization: 4-bit NF4 (BitsAndBytes)")
print(f"Deployment: 100% local, no cloud API")
print(f"Key benefit: Complete data privacy — no data leaves the machine")
print(f"Trade-off: Smaller model = lower quality vs cloud-based large models")

PART D SUMMARY
Model used: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Quantization: 4-bit NF4 (BitsAndBytes)
Deployment: 100% local, no cloud API
Key benefit: Complete data privacy — no data leaves the machine
Trade-off: Smaller model = lower quality vs cloud-based large models
